# 02. Agent Memory and Recall

Prerequisite: run `01_ingest_and_deduplicate.ipynb` first so this notebook can reuse the populated FalkorDB graph.

Sample data note:

- The tutorial texts in `notebooks/experimental_data/` are Medium.com articles by Filip Wojcik.
- Source profile: `https://medium.com/@filip.igor.wojcik`
- They are fully accessible without a subscription.

Install prerequisites before running the notebook:

```bash
uv sync --extra falkordblite --extra notebooks
```


In [ ]:
import os
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import autoroot  # noqa
from dotenv import load_dotenv
from pydantic_ai import Agent, RunContext
from rich import print

from grawiki.db import FalkorGraphDB
from grawiki.rag import GraphRAG

load_dotenv()


def locate_notebooks_dir() -> Path:
    cwd = Path.cwd().resolve()
    if (cwd / "experimental_data").exists():
        return cwd
    if (cwd / "notebooks" / "experimental_data").exists():
        return cwd / "notebooks"
    raise FileNotFoundError(
        "Could not locate the notebooks directory from the current working directory."
    )


NOTEBOOKS_DIR = locate_notebooks_dir()
DB_PATH = NOTEBOOKS_DIR / "local_falcor.db"
MODEL = (
    os.getenv("GRAWIKI_MODEL")
    or os.getenv("LLM_MODEL")
    or "openrouter/openai/gpt-5-mini"
)

AGENT_MODEL = "openrouter:openai/gpt-5-mini"

EMBEDDING_MODEL = (
    os.getenv("GRAWIKI_EMBEDDING_MODEL")
    or os.getenv("EMBEDDING_MODEL")
    or "openrouter:openai/text-embedding-3-small"
)

database = FalkorGraphDB(db_path=DB_PATH, graph_name="grawiki_notebooks")
rag = GraphRAG(model=MODEL, embedding_model=EMBEDDING_MODEL, db=database)


@dataclass
class NotebookDeps:
    rag: GraphRAG


def node_payload(node) -> dict[str, Any]:
    payload = node.model_dump()
    payload["labels"] = sorted(node.labels)
    return payload


def hit_payload(hit) -> dict[str, Any]:
    return {
        "score": round(hit.score, 4),
        "matched_on": hit.matched_on,
        "node": node_payload(hit.node),
    }


deps = NotebookDeps(rag=rag)
USER_ID = "demo-user"

In [ ]:
agent = Agent(
    model=AGENT_MODEL,
    deps_type=NotebookDeps,
    system_prompt=(
        "You are a notebook demo agent for GraWiki. Use search_knowledge_graph for facts from the graph, "
        "remember_user_fact when the user tells you something durable about themselves, and "
        "recall_user_memories before answering user-specific follow-up questions. "
        "You are assisting the user with id = 'demo-user'."
    ),
)


@agent.tool
async def search_knowledge_graph(
    ctx: RunContext[NotebookDeps],
    query: str,
    limit: int = 5,
) -> list[dict[str, Any]]:
    hits = await ctx.deps.rag.search(query, limit=limit)
    return [hit_payload(hit) for hit in hits]


@agent.tool
async def remember_user_fact(
    ctx: RunContext[NotebookDeps],
    memory_text: str,
    user_id: str,
    related_node_ids: list[str] | None = None,
) -> dict[str, Any]:
    memory = await ctx.deps.rag.remember(
        memory_text,
        metadata={"user_id": user_id},
        related_node_ids=related_node_ids or [],
    )
    return node_payload(memory)


@agent.tool
async def recall_user_memories(
    ctx: RunContext[NotebookDeps],
    query: str,
    user_id: str,
    hops: int = 1,
    limit: int = 5,
) -> list[dict[str, Any]]:
    hits = await ctx.deps.rag.recall(query, user_id=user_id, hops=hops, limit=limit)
    return [hit_payload(hit) for hit in hits]


print("Agent and tools are ready.")

## Demo conversation flow

The first call asks the agent to answer a factual question about the graph built in notebook 1.


In [ ]:
factual_result = await agent.run(
    "Using the knowledge graph, summarize the main idea behind connecting LLMs and knowledge graphs.",
    deps=deps,
)
print(factual_result.output)

In [ ]:
remember_result = await agent.run(
    """My name is Filip and I work as a researcher and data scientist. 
    I focus mostly on knowledge graphs and their integration with large language models.""",
    deps=deps,
)
print(remember_result.output)

In [ ]:
follow_up_result = await agent.run(
    "What do you remember about Filip and his interests?",
    deps=deps,
)
print(follow_up_result.output)

## Direct recall inspection

The facade-level recall API returns memory hits with readable graph context stored in `node.properties['recall_context']`.


In [ ]:
recall_hits = await rag.recall("Filip", hops=2, limit=5)
for hit in recall_hits:
    payload = hit_payload(hit)
    payload["recall_context"] = hit.node.properties.get("recall_context", "")
    print(payload)

In [ ]:
memory_rows = database.ro_query(
    "MATCH (m:__memory__) OPTIONAL MATCH (m)-[r]->(n) RETURN m.id, m.name, type(r), n.id, n.name ORDER BY m.creation_date, m.id LIMIT 20"
).result_set
print({"memory_rows": memory_rows})

# Reintegrate entities after memory saving session

In [ ]:
applied_reports = await rag.dedupe_entities(
    limit=5, threshold=0.9, min_merge_score=0.95, dry_run=False
)
entities_after_dedupe = await database.list_entities()

print({"entity_count_after_dedupe": len(entities_after_dedupe)})
for entity in entities_after_dedupe[:10]:
    print(
        {
            "id": entity.id,
            "name": entity.name,
            "labels": sorted(entity.labels),
            "semantic_key": entity.semantic_key,
        }
    )

In [ ]:
# Run this once when you are finished with the notebook session.
database.close()